# GToTR Tutorial 2: Brain Network Data Analysis

In this notebook, we work with brain network activity data that was studied using a Bayesian semiparametric model, which combines low-rank factorizations and flexible Gaussian process priors to learn changes in the conditional expectation of a network-valued random variable across the values of a continuous predictor, while including subject-specific random effects [1].

We will demonstrate how different assumptions on the responses and link function between the responses and the linear predictor in a GToTR model lead to different parameter estimators.

We will compute estimators using the following response distributions and link functions:
- **_Binomial distribution/Logit link_** (i.e., logistic regression)
- **_Gaussian distribution/Identity link_**

We will use special GToTR solvers to fit the above models.


[1] Lu Wang, Daniele Durante, Rex E. Jung, and David B. Dunson (2017), Bayesian network–response regression, 
_Bioinformatics_, 33(12):1859–1866. [DOI](http://doi.org/10.1093/bioinformatics/btx050)

-----

#### Notebook Outline

In this notebook, we will follow these steps:

1. Load the brain network data from files.
2. Examine the distribution of the responses.
3. Approximate $\mathcal{\widehat B}_{MLE}$ using a specialized Binomial/Logit solver in `gtotr`.
4. Approximate $\mathcal{\widehat B}_{MLE}$ using a specialized Gaussian/Identity solver in `gtotr`.
5. Compute estimated responses for both models: $\mathcal{\widehat Y}_i = \langle \mathcal{X}_i | \mathcal{\widehat B}_{MLE}\rangle$.
6. Plot the distributions of errors: $\mathcal{Y}_i - \mathcal{\widehat Y}_i$.
7. Plot slices of the estimated responses for the different models to compare to the original data.

-----

### Import Packages

In [ ]:
from __future__ import annotations

import gtotr

gtotr.__version__

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyttb as ttb

### Load the Data

In [ ]:
# read responses from file
Ys = ttb.import_data("data/brain_Ys.tensor")
print(f"responses shape: {Ys.shape}")

# read covariates from file
Xs = ttb.import_data("data/brain_Xs.tensor")
print(f"covariates shape: {Xs.shape}")

### Explore Response Data

In [ ]:
# plot an example slice of the response data
print("Ys[:,:,0]:")
plt.imshow(Ys[:, :, 0].data, cmap="gray", vmin=0, vmax=1)
plt.show()

In [ ]:
# plot histogram of values in responses
plt.hist(Ys.data.reshape((-1, 1)))
plt.show()
print(f"Unique Response Values: {set(Ys.data.flatten())}")

### Approximate $\mathcal{\widehat B}_{MLE}$ using Binomial/Logit GLM Solver

In [ ]:
model_binomial = gtotr.gtotr_cp(
    responses=Ys, covariates=Xs, family="binomial", link="logit"
)

In [ ]:
# Get the available methods for fitting this model
print(f"Fit methods: {model_binomial.fit_methods()}")

In [ ]:
# Use specialized solver for Gaussian family and Identity link
np.random.seed(123)
results_binomial = model_binomial.fit(
    method="cp_ao_glm", rank=5, tolerance=2e-3, printitn=1, maxiters=10
)
print(results_binomial.summary())

### Approximate $\mathcal{\widehat B}_{MLE}$ using Specialized Gaussian/Identity Solver

In [ ]:
# Use Fast GToTR specialized Gaussian/Identity solver
model_gaussian = gtotr.gtotr_cp(
    responses=Ys, covariates=Xs, family="gaussian", link="identity"
)

In [ ]:
# Get the available methods for fitting this model
print(f"Fit methods: {model_gaussian.fit_methods()}")

In [ ]:
# Use specialized solver for Gaussian family and Identity link
np.random.seed(123)
results_gaussian = model_gaussian.fit(
    method="cp_ao_gaussian_identity", rank=5, tolerance=2e-3, printitn=1, maxiters=10
)
print(results_gaussian.summary())

### Plot Estimated Response Errors for the Two Models

In [ ]:
# element-wise logistic function
def logistic_func(eta: np.ndarray, m: float = 1.0):
    """Element-wise logistic function."""
    return m / (1 + np.exp(-1.0 * eta))


Yhat_binomial = logistic_func(
    model_binomial.contract_xb(results_binomial.coef_).to_tensor().data
)
Yhat_gaussian = model_gaussian.contract_xb(results_gaussian.coef_).to_tensor().data

In [ ]:
# Plot errors
res_binomial = (Ys.data - Yhat_binomial).flatten()
plt.hist(res_binomial, bins=20, alpha=0.5, label="Binomial/Logit", color="blue")
rmse = np.sqrt(np.mean(res_binomial**2))
print(f"Binomial/Logit RMSE: {rmse:.4f}")

res_gaussian = (Ys.data - Yhat_gaussian).flatten()
plt.hist(res_gaussian, bins=21, alpha=0.5, label="Gaussian/Identity", color="orange")
rmse2 = np.sqrt(np.mean(res_gaussian**2))
print(f"Gaussian/Identity RMSE: {rmse2:.4f}")

plt.xlabel("Response Residuals (true - predicted)", fontsize=16)
plt.ylabel("Number of Samples", fontsize=16)

plt.legend()
plt.show()

### Plot Example Estimated Response for the Two Models

In [ ]:
# slice to plot
plot_slice = 0

fig, axes = plt.subplots(1, 3, figsize=(10, 5))
axes[0].imshow(Ys[:, :, plot_slice].data, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("True Response", fontsize=16)
axes[1].imshow(Yhat_binomial[:, :, plot_slice].data, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Binomial/Logit", fontsize=16)
axes[2].imshow(Yhat_gaussian[:, :, plot_slice].data, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Gaussian/Identity", fontsize=16)
plt.show()

-----

### Ideas for Further Investigation

1. What if you did not know that the responses were Gaussian distributed and wanted to use a different assumption (e.g., Poisson) in fitting the GToTR?